# SMS Spam Detection

SMS messages are classified as **spam** or **ham (not spam)** using machine learning.

Models compared:
- Naive Bayes
- Support Vector Machine (SVM)
- Decision Tree

Evaluation uses accuracy, precision, recall, F1-score, and confusion matrices.



## 1. Install Package

Install `kagglehub` to access the SMS Spam Collection dataset from Kaggle.

In [ ]:
!pip install kagglehub[pandas-datasets]

## 2. Download Dataset

Download the **SMS Spam Collection** dataset using `kagglehub`. The dataset contains SMS messages labeled as spam or ham.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Path to dataset files: /kaggle/input/sms-spam-collection-dataset


## 3. Check Dataset Files

List the downloaded files to confirm that the dataset is available.

In [ ]:
import os

print(os.listdir(path))

['spam.csv']


## 4. Load Dataset

Read the CSV file with Pandas using `latin-1` encoding and display the first five rows.

In [ ]:
import pandas as pd
import os

file_path = os.path.join(path, "spam.csv")

df = pd.read_csv(file_path, encoding="latin-1")

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## 5. Prepare Data

Only two columns are used:
- `label`: spam or ham category
- `message`: SMS message text

In [ ]:
df = df[["v1", "v2"]]

df.columns = ["label", "message"]

df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


## 6. Class Distribution

The dataset contains:
- **Ham:** 4,825 messages
- **Spam:** 747 messages

Ham messages are more common than spam messages, so the classes are imbalanced.

In [ ]:
df["label"].value_counts()

,count
label,
ham,4825
spam,747


## 7. Encode Labels

Convert the category labels to numerical values:
- `0`: Ham
- `1`: Spam

In [ ]:
df["label_num"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

df.head()

,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 8. Features and Train–Test Split

`X` contains the message texts, while `y` contains the numerical labels. The dataset is split into **80% training** and **20% testing** data. `stratify=y` preserves the class proportions.

In [ ]:
X = df["message"]
y = df["label_num"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## 9. TF-IDF Feature Extraction

TF-IDF converts the message texts into numerical features based on word importance. Common English stop words are removed. The vectorizer is fitted only on the training data and then applied to the test data.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english")

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## 10. Model Training and Evaluation

Three models are trained on TF-IDF features:
- **Multinomial Naive Bayes:** probabilistic classification
- **Linear SVM:** classification using a separating hyperplane
- **Decision Tree:** rule-based classification through feature splits

Performance is measured using the test data.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    "Naive Bayes": MultinomialNB(),
    "SVM": LinearSVC(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

for model_name, model in models.items():
    print("\n==============================")
    print("Model:", model_name)
    print("==============================")

    model.fit(X_train_tfidf, y_train)

    y_pred = model.predict(X_test_tfidf)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))


Model: Naive Bayes
Accuracy: 0.968609865470852

Classification Report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115


Confusion Matrix:
[[966   0]
 [ 35 114]]

Model: SVM
Accuracy: 0.9838565022421525

Classification Report:
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115


Confusion Matrix:
[[965   1]
 [ 17 132]]

Model: Decision Tree
Accuracy: 0.9596412556053812

Classification Report:
              precision    recall  f1-score   supp

## 11. Model Comparison

| Model | Accuracy | Spam F1-score |
|---|---:|---:|
| Naive Bayes | 96.86% | 0.87 |
| SVM | **98.39%** | **0.94** |
| Decision Tree | 95.96% | 0.84 |

**SVM** has the highest accuracy and spam F1-score in the recorded evaluation. It correctly identifies **132 of 149 spam messages** and misses **17** spam messages.

## 12. Predict New Messages

The selected SVM model predicts the labels of new SMS messages. Each message is transformed using the same fitted TF-IDF vectorizer before prediction.

In [ ]:
best_model = LinearSVC()

best_model.fit(X_train_tfidf, y_train)

new_message = ["Congratulations! You won a free prize. Click this link now"]

new_message_tfidf = vectorizer.transform(new_message)

prediction = best_model.predict(new_message_tfidf)

print("Pesan:", new_message[0])

if prediction[0] == 1:
    print("Prediksi: SPAM")
else:
    print("Prediksi: HAM / bukan spam")

Pesan: Congratulations! You won a free prize. Click this link now
Prediksi: SPAM


In [ ]:
new_message = ["Are we still meeting at campus tomorrow?"]

new_message_tfidf = vectorizer.transform(new_message)

prediction = best_model.predict(new_message_tfidf)

print("Pesan:", new_message[0])

if prediction[0] == 1:
    print("Prediksi: SPAM")
else:
    print("Prediksi: HAM / bukan spam")

Pesan: Are we still meeting at campus tomorrow?
Prediksi: HAM / bukan spam


## 13. Test Multiple Messages

Ten sample messages are classified as spam or ham. The promotional message **"Limited offer! Buy now and get 50% discount"** is predicted as ham in the recorded output.

In [ ]:
test_messages = [
    "Congratulations! You won a free prize. Click this link now",
    "Are we still meeting at campus tomorrow?",
    "URGENT! Your account has been suspended. Click the link to verify now",
    "Hey, don't forget to bring your laptop for class today",
    "You have won $1000 cash. Call this number to claim your prize",
    "Can you send me the assignment file tonight?",
    "Free entry in a weekly competition. Text WIN to 80086 now",
    "Mom said dinner is ready, come home soon",
    "Limited offer! Buy now and get 50% discount",
    "Please remind me about the group meeting at 3 PM"
]

test_messages_tfidf = vectorizer.transform(test_messages)

predictions = best_model.predict(test_messages_tfidf)

for message, prediction in zip(test_messages, predictions):
    print("Pesan:", message)

    if prediction == 1:
        print("Prediksi: SPAM")
    else:
        print("Prediksi: HAM / bukan spam")

    print("-" * 80)

Pesan: Congratulations! You won a free prize. Click this link now
Prediksi: SPAM
--------------------------------------------------------------------------------
Pesan: Are we still meeting at campus tomorrow?
Prediksi: HAM / bukan spam
--------------------------------------------------------------------------------
Pesan: URGENT! Your account has been suspended. Click the link to verify now
Prediksi: SPAM
--------------------------------------------------------------------------------
Pesan: Hey, don't forget to bring your laptop for class today
Prediksi: HAM / bukan spam
--------------------------------------------------------------------------------
Pesan: You have won $1000 cash. Call this number to claim your prize
Prediksi: SPAM
--------------------------------------------------------------------------------
Pesan: Can you send me the assignment file tonight?
Prediksi: HAM / bukan spam
--------------------------------------------------------------------------------
Pesan: Free en

## 14. Conclusion

- All three algorithms were trained and evaluated for SMS spam classification.
- **SVM** achieved the best recorded accuracy at **98.39%**.
- TF-IDF converted the SMS texts into numerical features.
- Some spam messages were still misclassified as ham.